In [1]:
import json
from pathlib import Path

DATA_DIR = Path("data")

## Task 2.1: Recursive search for log files

The function walks through the folder structure using `Path.iterdir()`. If an item is a folder, the function calls itself on that folder. If it is a file ending in `.log`, it is added to the list.

In [2]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    """Recursively find all files with the given extension.

    Uses explicit recursion with Path.iterdir(), not glob, rglob or os.walk.
    """
    directory = Path(directory)
    found_files: list[Path] = []

    for item in directory.iterdir():
        if item.is_dir():
            found_files.extend(find_log_files(item, extension))
        elif item.is_file() and item.suffix == extension:
            found_files.append(item)

    return found_files


log_files = find_log_files(DATA_DIR)
print(f"Found {len(log_files)} log files:")
for path in sorted(log_files):
    print(path)

Found 9 log files:
data/2026/library_cafe/POS_01/register_sales.log
data/2026/library_cafe/POS_02/register_sales.log
data/2026/library_cafe/POS_03/register_sales.log
data/2026/science_building/POS_01/register_sales.log
data/2026/science_building/POS_02/register_sales.log
data/2026/science_building/POS_03/register_sales.log
data/2026/student_union/POS_01/register_sales.log
data/2026/student_union/POS_02/register_sales.log
data/2026/student_union/POS_03/register_sales.log


**Result:** The function finds 9 log files, one for each of the three
registers in all three stores. Note that `student_union` has log files,
but is not listed in `store_config.json`. We return to this in Task 2.2.

## Task 2.2: Recursive search for active registers

The function handles three cases. If the input is a dictionary, it checks whether `"active"` is `True` and stores the `"id"`, then searches all values. If the input is a list, it searches each element. Anything else ends the recursion and returns an empty set.

In [3]:
def find_active_registers(data: object) -> set[str]:
    """Recursively search nested dicts and lists for registers
    where "active" is True, and return their ids.
    """
    active_ids: set[str] = set()

    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_ids.add(data["id"])
        for value in data.values():
            active_ids |= find_active_registers(value)

    elif isinstance(data, list):
        for element in data:
            active_ids |= find_active_registers(element)

    return active_ids


with open(DATA_DIR / "store_config.json", encoding="utf-8") as f:
    config = json.load(f)

active_registers = find_active_registers(config)
print(active_registers)

{'POS_02', 'POS_03', 'POS_01'}


**Result:** `POS_04` is correctly excluded since it is inactive. However, the set only stores register IDs, not which store they belong to. Since `POS_01` to `POS_03` exist in all three stores in the log files, filtering on ID alone would also keep logs from `student_union`, which is not in the config at all.